In [2]:
"""
STEP 5.4 (supplementary) - MODEL RESPONSE-CURVE AUDIT
=====================================================

Why this step exists
--------------------
Step 4.3 / 5.1 showed that XGBoost has the best error in the Normal regime but
collapses in the Critical regime (> 233.68 min: MAE 52.68 vs ~25 min for Ridge/RF).
This script probes the four saved models directly to show *why*: it sweeps the
current-stop delay while holding every other feature at its training median and
records each model's forecast for the next stop (a partial-dependence style
"response curve").

It needs only the artefacts that are committed to the repository:
    results/models/step04_03_linear_regression.joblib
    results/models/step04_03_random_forest.joblib
    results/models/step04_03_xgboost.json
    results/step04_02_preprocessing_artifacts.json

Outputs
-------
    results/step05_04_model_response_curves.json
    results/figures/step05_04_model_response_curves.png

Run from the repository root:
    python analysis/step05_04_model_response_audit.py
"""

import json
import warnings
from pathlib import Path

import joblib
import numpy as np

warnings.filterwarnings("ignore")  # sklearn version-mismatch warnings on unpickle

BASE_DIR = Path.cwd().parent
RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = RESULT_DIR / "models"
FIG_DIR = RESULT_DIR / "figures"

meta = json.loads((RESULT_DIR / "step04_02_preprocessing_artifacts.json").read_text())
FEATURES = meta["feature_columns"]
SCALER = meta["standard_scaler_fit_on_train"]
MU = np.array([SCALER[c]["mean"] for c in FEATURES], dtype=np.float64)
SD = np.array([SCALER[c]["std"] for c in FEATURES], dtype=np.float64)
T2SD = 167.653054
T3SD = 233.67987

lin = joblib.load(MODEL_DIR / "step04_03_linear_regression.joblib")
rf = joblib.load(MODEL_DIR / "step04_03_random_forest.joblib")
try:
    from xgboost import XGBRegressor

    xgb = XGBRegressor()
    xgb.load_model((MODEL_DIR / "step04_03_xgboost.json").as_posix())
except Exception as exc:  # pragma: no cover
    xgb = None
    print(f"XGBoost unavailable ({exc}); its curve will be skipped.")


def build_matrix(delays, scenario):
    base = {c: SCALER[c]["median"] for c in FEATURES}
    if scenario == "terminal":
        base["is_terminal_arrival"] = 1.0
        base["route_progress_ratio"] = 1.0
    rows = []
    for d in delays:
        f = dict(base)
        f["curr_delay_min"] = float(d)
        f["prev_delay_min"] = float(d)  # zero upstream momentum
        f["prev_delay_delta_min"] = 0.0
        rows.append([f[c] for c in FEATURES])
    return np.asarray(rows, dtype=np.float32)


delays = np.arange(-30, 1441, 5, dtype=np.float64)
curves = {"curr_delay_min": delays.tolist(), "scenarios": {}}

for scenario in ["intermediate", "terminal"]:
    X = build_matrix(delays, scenario)
    out = {
        "1_Persistence_Baseline": delays.tolist(),
        "2_Linear_Regression": lin.predict((X - MU) / SD).round(2).tolist(),
        "3_Random_Forest": rf.predict(X).round(2).tolist(),
    }
    if xgb is not None:
        out["4_XGBoost"] = xgb.predict(X).astype(float).round(2).tolist()
    curves["scenarios"][scenario] = out

# Quantify the XGBoost plateau: largest current delay that still moves the forecast
summary = {}
if xgb is not None:
    xg = np.array(curves["scenarios"]["intermediate"]["4_XGBoost"])
    tail = delays >= T3SD
    summary["xgb_forecast_range_in_critical_tail_min"] = [float(xg[tail].min()), float(xg[tail].max())]
    last_change = float(delays[np.where(np.abs(np.diff(xg)) > 0.5)[0].max() + 1])
    summary["xgb_forecast_constant_above_curr_delay_min"] = last_change
    summary["xgb_plateau_value_min"] = float(xg[-1])
    for name in ["2_Linear_Regression", "3_Random_Forest", "4_XGBoost"]:
        y = np.array(curves["scenarios"]["intermediate"][name])
        summary[f"{name}_abs_gap_vs_identity_at_1000min"] = float(abs(y[delays == 1000][0] - 1000))

# Effective (raw-unit) linear coefficients so the ridge model can be re-used anywhere
eff_coef = (lin.coef_ / SD)
eff_intercept = float(lin.intercept_ - np.sum(lin.coef_ * MU / SD))
curves["ridge_raw_unit_equation"] = {
    "intercept": round(eff_intercept, 6),
    "coefficients": {c: round(float(v), 6) for c, v in zip(FEATURES, eff_coef)},
    "note": "y_hat = intercept + sum(coef * raw_feature). Equivalent to Ridge(alpha=10) on z-scored features.",
}
curves["summary"] = summary
curves["held_constant_at_training_median"] = {
    c: SCALER[c]["median"] for c in FEATURES if c not in ("curr_delay_min", "prev_delay_min", "prev_delay_delta_min")
}

(RESULT_DIR / "step05_04_model_response_curves.json").write_text(json.dumps(curves, indent=2))
print(json.dumps(summary, indent=2))

# ---------- figure ----------
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

colors = {
    "1_Persistence_Baseline": "#8a8a8a",
    "2_Linear_Regression": "#2a6fdb",
    "3_Random_Forest": "#1f9e6e",
    "4_XGBoost": "#e0611f",
}
labels = {
    "1_Persistence_Baseline": "Persistence (identity)",
    "2_Linear_Regression": "Ridge regression",
    "3_Random_Forest": "Random forest",
    "4_XGBoost": "XGBoost (hist)",
}
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), dpi=150)
for ax, (scenario, title) in zip(
    axes, [("intermediate", "Intermediate stop"), ("terminal", "Terminal destination")]
):
    for name, ys in curves["scenarios"][scenario].items():
        ax.plot(delays, ys, label=labels[name], color=colors[name],
                lw=2.4 if name == "4_XGBoost" else 1.8,
                ls="--" if name == "1_Persistence_Baseline" else "-")
    ax.axvspan(T3SD, 1440, color="#c0392b", alpha=0.06)
    ax.axvline(T2SD, color="#999", lw=0.8, ls=":")
    ax.axvline(T3SD, color="#c0392b", lw=0.8, ls=":")
    ax.text(T3SD + 15, 60, "Critical regime (> 3 SD)", color="#c0392b", fontsize=9)
    ax.set_title(f"{title}: forecast vs. current delay", fontsize=11, fontweight="bold")
    ax.set_xlabel("Current-stop delay (min); previous delay = current; other features at training median")
    ax.set_ylabel("Forecast next-stop delay (min)")
    ax.set_xlim(-30, 1440)
    ax.set_ylim(-30, 1440)
    ax.grid(alpha=0.3)
axes[0].legend(loc="upper left", frameon=False)
fig.suptitle("Model response-curve audit: XGBoost forecasts saturate in the critical tail", fontsize=12, fontweight="bold")
fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "step05_04_model_response_curves.png", bbox_inches="tight")
print("Saved results/figures/step05_04_model_response_curves.png")


{
  "xgb_forecast_range_in_critical_tail_min": [
    229.55,
    689.06
  ],
  "xgb_forecast_constant_above_curr_delay_min": 435.0,
  "xgb_plateau_value_min": 689.06,
  "2_Linear_Regression_abs_gap_vs_identity_at_1000min": 3.340000000000032,
  "3_Random_Forest_abs_gap_vs_identity_at_1000min": 14.92999999999995,
  "4_XGBoost_abs_gap_vs_identity_at_1000min": 310.94000000000005
}
Saved results/figures/step05_04_model_response_curves.png
